<br/>
<img src="images/LOGO-CORREO_x.png" alt="SkillNest" width="280px" align="left"/>
<img src="images/LOGO-CORREO_x.png" alt="SkillNest" width="280px" align="right"/>
<div align="center">
<h2>Bootcamp Data Science — Módulo 2</h2><br/>
<h1>Semana 8 · Lunes — Optimización de Hiperparámetros</h1>
<h3>GridSearchCV, RandomizedSearchCV, LazyPredict y Optuna</h3>
<br/>
    <b>Instructor:</b> Jesús Ortiz · jesus.jeduardo7@gmail.com<br/><br/>
    <b>SkillNest · b2b-sonda-data-science</b>
</div>
<br/>

## Objetivos

Al final de la clase van a poder:

1. Entender la diferencia entre GridSearchCV y RandomizedSearchCV y cuándo usar cada uno.
2. Usar LazyPredict para descartar rápido modelos que no prometen.
3. Usar Optuna para hacer búsqueda bayesiana (la opción profesional).
4. Resolver UN ejercicio largo y exigente comparando los 3 métodos en un dataset desafiante.

# 1. El problema: ¿cómo elegimos hiperparámetros?

Hasta ahora hemos hecho dos cosas: probar manualmente algunos valores o usar GridSearchCV. Pero en problemas reales tenemos modelos con 8-15 hiperparámetros, y cada uno con 4-5 valores razonables. Eso son **miles de combinaciones**. Probarlas todas no es viable.

Las opciones son:

| Método | Cómo busca | Cuándo usarlo |
|---|---|---|
| **GridSearchCV** | Exhaustivo: prueba todas las combinaciones | Pocas combinaciones (< 100) |
| **RandomizedSearchCV** | Aleatorio: muestrea N combinaciones | Cuando hay muchas combinaciones |
| **LazyPredict** | Entrena ~30 modelos con defaults | Para descartar modelos rápido |
| **Optuna** | Búsqueda bayesiana inteligente | Para optimización seria |

Cada uno tiene su lugar. Vamos a verlos en orden.

## Setup común

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', None)

# Dataset: California Housing
data = fetch_california_housing(as_frame=True)
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')

# 2. GridSearchCV: la búsqueda exhaustiva

GridSearchCV prueba TODAS las combinaciones posibles. Si tienen un grid de 3 × 3 × 4 = 36 combinaciones con cv=5, entrenan 180 modelos.

Funciona bien cuando:
- Tenemos pocas combinaciones (menos de 100).
- Sabemos bien qué rango de valores explorar.
- Tenemos tiempo y máquina.

In [ ]:
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth':    [5, 10, 20, None],
    'min_samples_leaf': [1, 3, 5]
}
# 3 × 4 × 3 = 36 combinaciones × 5 folds = 180 entrenamientos

t0 = time.time()
grid = GridSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    param_grid, cv=5, scoring='r2', n_jobs=-1
).fit(X_train, y_train)
t_grid = time.time() - t0

print(f'Mejores hiperparámetros: {grid.best_params_}')
print(f'Mejor R² (CV): {grid.best_score_:.4f}')
print(f'R² en test:    {grid.score(X_test, y_test):.4f}')
print(f'Tiempo total:  {t_grid:.1f} segundos')

# 3. RandomizedSearchCV: muestreo aleatorio

En vez de probar todas las combinaciones, muestrea N al azar. Suena loco pero funciona porque normalmente solo unos pocos hiperparámetros importan en cada problema. El paper de Bergstra y Bengio (2012) mostró que random search es igual o mejor que grid search cuando hay muchos hiperparámetros.

Ventajas:
- Mucho más rápido si limitamos el número de iteraciones.
- Permite usar distribuciones continuas (no solo valores discretos).
- Es lo recomendado cuando hay más de 4-5 hiperparámetros.

In [ ]:
from scipy.stats import randint

param_dist = {
    'n_estimators':     randint(50, 500),
    'max_depth':        randint(3, 30),
    'min_samples_leaf': randint(1, 10),
    'min_samples_split': randint(2, 20),
    'max_features':     [0.5, 0.7, 1.0, 'sqrt']
}

t0 = time.time()
rand = RandomizedSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    param_dist, n_iter=30, cv=5, scoring='r2',
    n_jobs=-1, random_state=42
).fit(X_train, y_train)
t_rand = time.time() - t0

print(f'Mejores hiperparámetros: {rand.best_params_}')
print(f'Mejor R² (CV): {rand.best_score_:.4f}')
print(f'R² en test:    {rand.score(X_test, y_test):.4f}')
print(f'Tiempo total:  {t_rand:.1f} segundos')
print(f'\nComparación:')
print(f'  GridSearch:       180 entrenamientos, {t_grid:.0f}s, R² test = {grid.score(X_test, y_test):.4f}')
print(f'  RandomizedSearch: 150 entrenamientos, {t_rand:.0f}s, R² test = {rand.score(X_test, y_test):.4f}')

# 4. LazyPredict: descartar modelos rápido

LazyPredict entrena ~30 modelos con sus parámetros por defecto y los compara. NO sirve para producción, sirve para descartar familias de modelos que no prometen antes de invertir tiempo tuneando.

Instalación: `pip install lazypredict`

Estrategia recomendada:
1. Correr LazyPredict para ver qué familias prometen.
2. Tomar los 2-3 mejores.
3. Tunearlos con RandomizedSearch o Optuna.

In [ ]:
try:
    from lazypredict.Supervised import LazyRegressor
    
    # Usamos una muestra del train para que sea rápido
    np.random.seed(42)
    idx = np.random.choice(len(X_train), size=3000, replace=False)
    
    reg = LazyRegressor(verbose=0, ignore_warnings=True)
    modelos, _ = reg.fit(X_train.iloc[idx], X_test, y_train.iloc[idx], y_test)
    print('Top 10 modelos según LazyPredict:')
    print(modelos.head(10))
except ImportError:
    print('LazyPredict no instalado. Corre: !pip install lazypredict')

# 5. Optuna: la búsqueda bayesiana

Optuna es la herramienta profesional para optimización de hiperparámetros. A diferencia de GridSearch y RandomSearch, NO prueba al azar: usa los resultados anteriores para decidir qué probar después. Eso se llama **Tree-structured Parzen Estimator (TPE)** y es básicamente machine learning para tunear modelos.

Ventajas:
- Encuentra mejores hiperparámetros con menos pruebas que RandomSearch.
- Permite definir espacios complejos (rangos continuos, categóricos, condicionales).
- Tiene visualizaciones muy buenas para entender qué hiperparámetros importan.
- Permite paralelización y early stopping.

Instalación: `pip install optuna`

In [ ]:
try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    
    def objective(trial):
        params = {
            'n_estimators':     trial.suggest_int('n_estimators', 50, 500),
            'max_depth':        trial.suggest_int('max_depth', 3, 30),
            'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
            'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
            'max_features':     trial.suggest_float('max_features', 0.3, 1.0)
        }
        modelo = RandomForestRegressor(**params, random_state=42, n_jobs=-1)
        # Usamos un cv=3 para que sea más rápido (en producción cv=5)
        return cross_val_score(modelo, X_train, y_train, cv=3, scoring='r2', n_jobs=-1).mean()
    
    t0 = time.time()
    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=30, show_progress_bar=False)
    t_optuna = time.time() - t0
    
    print(f'Mejores hiperparámetros: {study.best_params}')
    print(f'Mejor R² (CV): {study.best_value:.4f}')
    print(f'Tiempo total:  {t_optuna:.1f} segundos')
    
    # Entrenar modelo final con los mejores y evaluar en test
    mejor_modelo = RandomForestRegressor(**study.best_params, random_state=42, n_jobs=-1).fit(X_train, y_train)
    print(f'R² en test:    {mejor_modelo.score(X_test, y_test):.4f}')
except ImportError:
    print('Optuna no instalado. Corre: !pip install optuna')

# 6. Comparación rápida

| Método | Entrenamientos | Cuándo conviene |
|---|---|---|
| GridSearch | 180 | Pocas combinaciones, sé qué rango buscar |
| RandomSearch | 150 | Muchas combinaciones, exploración rápida |
| LazyPredict | ~30 modelos default | Descartar familias antes de tunear |
| Optuna | ~30-100 | Optimización seria, búsqueda inteligente |

Mi recomendación práctica: empezar con LazyPredict para tener un baseline rápido, después usar Optuna sobre los 2-3 mejores modelos.

---
# Ejercicio integrador

Un solo ejercicio, pero exigente. La idea es que apliquen los 3 métodos y comparen.

## Reto: predecir el precio mediano de casas con 3 estrategias de tuning

Dataset: `fetch_california_housing` (el que usamos en la demo).

Modelo base: `RandomForestRegressor` (lo conocemos bien de las semanas 6 y 7).

### Lo que tienen que hacer

**Parte A — Baseline sin tuning (5 min)**
1. Entrenar RandomForestRegressor con parámetros por defecto.
2. Reportar R² test, MAE test, tiempo de entrenamiento.

**Parte B — GridSearchCV (15 min)**
3. Definir un grid pequeño (máximo 4 × 3 × 3 = 36 combinaciones).
4. Correr GridSearchCV con cv=5.
5. Reportar mejores hiperparámetros, R² test, MAE test, tiempo de entrenamiento.

**Parte C — RandomizedSearchCV (15 min)**
6. Definir distribuciones más amplias (n_estimators 50-500, max_depth 3-30, etc).
7. Correr RandomizedSearch con n_iter=30, cv=5.
8. Reportar mejores hiperparámetros, R² test, MAE test, tiempo de entrenamiento.

**Parte D — Optuna (20 min)**
9. Definir un objective con los mismos hiperparámetros que en Random Search.
10. Correr 30 trials.
11. Reportar mejores hiperparámetros, R² test, MAE test, tiempo de entrenamiento.
12. Graficar `optuna.visualization.plot_optimization_history(study)` para ver cómo mejoró el modelo trial a trial.
13. Graficar `optuna.visualization.plot_param_importances(study)` para ver qué hiperparámetros importan más.

**Parte E — Tabla comparativa y decisión final**
14. Hacer una tabla con las 4 estrategias (Baseline, Grid, Random, Optuna) comparando R², MAE y tiempo.
15. ¿Cuál ganó? ¿La diferencia justifica el tiempo extra?
16. Si tuvieran que entregar UN modelo a producción, ¿cuál elegirían? Justifiquen.

### Bonus que vale puntos extra

- Comparar Optuna con n_trials=10 vs n_trials=50: ¿cuánto mejora con más trials?
- Probar Optuna pero optimizando MAE (en vez de R²): ¿cambia el ganador?
- Investigar y aplicar `pruners` de Optuna para descartar trials que no prometen rápido.

In [ ]:
# Parte A — Baseline



In [ ]:
# Parte B — GridSearchCV



In [ ]:
# Parte C — RandomizedSearchCV



In [ ]:
# Parte D — Optuna



In [ ]:
# Parte E — Tabla comparativa y decisión final



## Cierre

Hoy aprendimos:

- GridSearchCV exhaustivo: bueno cuando tenemos pocas combinaciones.
- RandomizedSearchCV: mucho más rápido y casi igual de bueno cuando hay muchos hiperparámetros.
- LazyPredict para descartar familias de modelos antes de tunear.
- Optuna como la opción profesional: búsqueda bayesiana inteligente.

Mañana vamos con LightGBM, un modelo de boosting que vamos a tunear con Optuna usando lo que aprendimos hoy.